In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
import warnings
warnings.filterwarnings('ignore')


# ==========================================
# 1. 데이터 로드 (메모리 최적화)
# ==========================================
def load_data(train_path='train.csv', test_path=None):
    feature_cols = [str(i) for i in range(226)]
    target_cols = ['layer_1', 'layer_2', 'layer_3', 'layer_4']

    dtypes = {col: 'float32' for col in feature_cols}
    for col in target_cols:
        dtypes[col] = 'float32'

    print("[1/4] train.csv 로딩 중...")
    train_df = pd.read_csv(train_path, dtype=dtypes)

    test_df = None
    if test_path is not None:
        test_dtypes = {col: 'float32' for col in feature_cols}
        test_df = pd.read_csv(test_path, dtype=test_dtypes)

    return train_df, test_df, feature_cols, target_cols


# ==========================================
# 2. 건너뛰기 분할 (Interleaved Split)
# ==========================================
def interleaved_split(df, target_cols, valid_ratio=0.1, seed=42):
    print("[2/4] 건너뛰기 분할(Interleaved Split) 적용 중...")
    Y = df[target_cols].values.astype(int)

    has_300 = (Y == 300).any(axis=1)
    is_train_thickness = ((Y // 10) % 2 == 1) & (Y < 300)
    train_mask = is_train_thickness.all(axis=1)

    eval_pool_mask = (~train_mask) & (~has_300)
    extrap_mask = has_300

    eval_indices = np.where(eval_pool_mask)[0]
    rng = np.random.default_rng(seed)
    rng.shuffle(eval_indices)

    n_valid = int(len(eval_indices) * valid_ratio)
    valid_idx = eval_indices[:n_valid]
    test_idx = eval_indices[n_valid:]
    train_idx = np.where(train_mask)[0]
    extrap_idx = np.where(extrap_mask)[0]

    is_test_layer = ((Y // 10) % 2 == 0) & (Y < 300)
    num_unseen_layers = is_test_layer.sum(axis=1)

    print(f"  - Train 샘플 수: {len(train_idx):,}개 (비율: {len(train_idx)/len(df)*100:.1f}%)")
    print(f"  - Valid 샘플 수: {len(valid_idx):,}개 (비율: {len(valid_idx)/len(df)*100:.1f}%)")
    print(f"  - Test  샘플 수: {len(test_idx):,}개 (비율: {len(test_idx)/len(df)*100:.1f}%)")
    print(f"  - 외삽(300nm) 수: {len(extrap_idx):,}개 (비율: {len(extrap_idx)/len(df)*100:.1f}%)")

    return train_idx, valid_idx, test_idx, extrap_idx, num_unseen_layers


# ==========================================
# 3. 도메인 + 신호처리 피처 엔지니어링
# ==========================================
def extract_domain_features(R_raw, n_fft_modes=32, use_smoothed_spectrum=True):
    print("[3/4] 반도체 광학·주파수 도메인 피처 추출 중...")
    N, L = R_raw.shape
    feats = []
    feat_names = []

    # ① Savitzky-Golay 노이즈 스무딩
    R_smooth = savgol_filter(R_raw, window_length=9, polyorder=3, axis=1).astype(np.float32)
    noise_residual = R_raw - R_smooth
    noise_std = np.std(noise_residual, axis=1, keepdims=True)

    if use_smoothed_spectrum:
        feats.append(R_smooth)
        feat_names.extend([f"smooth_{i}" for i in range(L)])
    else:
        feats.append(R_raw)
        feat_names.extend([f"raw_{i}" for i in range(L)])

    feats.append(noise_std)
    feat_names.append("noise_residual_std")

    # ② 푸리에 변환(FFT) 주파수 피처
    R_centered = R_smooth - np.mean(R_smooth, axis=1, keepdims=True)
    fft_complex = np.fft.rfft(R_centered, axis=1)

    fft_modes = fft_complex[:, 1:n_fft_modes + 1]
    fft_amp = np.abs(fft_modes).astype(np.float32) / L
    fft_real = np.real(fft_modes).astype(np.float32) / L
    fft_imag = np.imag(fft_modes).astype(np.float32) / L

    dom_freq_idx = (np.argmax(fft_amp, axis=1, keepdims=True) + 1).astype(np.float32)
    max_freq_amp = np.max(fft_amp, axis=1, keepdims=True)
    freq_bins = np.arange(1, n_fft_modes + 1, dtype=np.float32).reshape(1, -1)
    spectral_centroid = np.sum(fft_amp * freq_bins, axis=1, keepdims=True) / (
        np.sum(fft_amp, axis=1, keepdims=True) + 1e-8
    )

    feats.extend([fft_amp, fft_real, fft_imag, dom_freq_idx, max_freq_amp, spectral_centroid])
    feat_names.extend([f"fft_amp_m{i}" for i in range(1, n_fft_modes + 1)])
    feat_names.extend([f"fft_real_m{i}" for i in range(1, n_fft_modes + 1)])
    feat_names.extend([f"fft_imag_m{i}" for i in range(1, n_fft_modes + 1)])
    feat_names.extend(["fft_dominant_mode", "fft_max_amp", "fft_spectral_centroid"])

    # ③ 파장 대역별 물리 통계량
    bands = {
        "full": R_smooth,
        "vis": R_smooth[:, :125],
        "uv": R_smooth[:, 125:],
        "si_e1": R_smooth[:, 125:155],
        "deep_uv": R_smooth[:, 155:]
    }

    for b_name, b_data in bands.items():
        b_mean = np.mean(b_data, axis=1, keepdims=True)
        b_std = np.std(b_data, axis=1, keepdims=True)
        b_ptp = np.ptp(b_data, axis=1, keepdims=True)
        feats.extend([b_mean, b_std, b_ptp])
        feat_names.extend([f"{b_name}_mean", f"{b_name}_std", f"{b_name}_ptp"])

    uv_vis_mean_ratio = np.mean(bands["uv"], axis=1, keepdims=True) / (
        np.mean(bands["vis"], axis=1, keepdims=True) + 1e-5
    )
    uv_vis_std_ratio = np.std(bands["uv"], axis=1, keepdims=True) / (
        np.std(bands["vis"], axis=1, keepdims=True) + 1e-5
    )
    feats.extend([uv_vis_mean_ratio, uv_vis_std_ratio])
    feat_names.extend(["uv_vis_mean_ratio", "uv_vis_std_ratio"])

    # ④ 스펙트럼 1·2차 미분 및 극점 개수
    d1 = np.diff(R_smooth, n=1, axis=1)
    d2 = np.diff(R_smooth, n=2, axis=1)

    d1_abs_mean = np.mean(np.abs(d1), axis=1, keepdims=True)
    d1_std = np.std(d1, axis=1, keepdims=True)
    d2_abs_mean = np.mean(np.abs(d2), axis=1, keepdims=True)

    d1_vis_energy = np.mean(np.abs(d1[:, :124]), axis=1, keepdims=True)
    d1_uv_energy = np.mean(np.abs(d1[:, 124:]), axis=1, keepdims=True)

    sign_changes = ((d1[:, :-1] * d1[:, 1:]) < 0).sum(axis=1, keepdims=True).astype(np.float32)

    feats.extend([d1_abs_mean, d1_std, d2_abs_mean, d1_vis_energy, d1_uv_energy, sign_changes])
    feat_names.extend([
        "d1_abs_mean", "d1_std", "d2_abs_mean",
        "d1_vis_energy", "d1_uv_energy", "num_extrema_peaks"
    ])

    X_fe = np.hstack(feats).astype(np.float32)
    print(f"  - 피처 생성 완료: 원본 {L}개 -> 최종 {X_fe.shape[1]}개 특성")
    return X_fe, feat_names


# ==========================================
# 4. 성능 평가 함수
# ==========================================
def evaluate_predictions(y_true, y_pred, model_name):
    layer_maes = [mean_absolute_error(y_true[:, i], y_pred[:, i]) for i in range(4)]
    total_mae = np.mean(layer_maes)
    print(
        f"[{model_name}] 전체 MAE: {total_mae:.3f} nm | "
        f"L1(SiN): {layer_maes[0]:.2f}, L2(SiO2): {layer_maes[1]:.2f}, "
        f"L3(SiN): {layer_maes[2]:.2f}, L4(SiO2): {layer_maes[3]:.2f}"
    )
    return total_mae, layer_maes


# ==========================================
# 5. 실행 진입점
# ==========================================
if __name__ == "__main__":
    TRAIN_PATH = "/content/drive/MyDrive/CUAI_project/train.csv"

    train_df, _, feature_cols, target_cols = load_data(TRAIN_PATH)

    R_raw = train_df[feature_cols].values
    Y_all = train_df[target_cols].values

    train_idx, valid_idx, test_idx, extrap_idx, num_unseen = interleaved_split(train_df, target_cols)
    X_fe, feat_names = extract_domain_features(R_raw, n_fft_modes=32)

    print("\n[4/4] 건너뛰기 분할 Valid 셋 기준 피처 엔지니어링 효과 검증...")

    mean_pred = np.full_like(Y_all[valid_idx], 155.0)
    evaluate_predictions(Y_all[valid_idx], mean_pred, "0. 평균 예측(155nm)")

    scaler_raw = StandardScaler()
    X_tr_raw = scaler_raw.fit_transform(R_raw[train_idx])
    X_va_raw = scaler_raw.transform(R_raw[valid_idx])

    ridge_raw = Ridge(alpha=1.0).fit(X_tr_raw, Y_all[train_idx])
    evaluate_predictions(Y_all[valid_idx], ridge_raw.predict(X_va_raw), "1-A. Ridge (원본 226개)")

    scaler_fe = StandardScaler()
    X_tr_fe = scaler_fe.fit_transform(X_fe[train_idx])
    X_va_fe = scaler_fe.transform(X_fe[valid_idx])

    ridge_fe = Ridge(alpha=1.0).fit(X_tr_fe, Y_all[train_idx])
    evaluate_predictions(Y_all[valid_idx], ridge_fe.predict(X_va_fe), "1-B. Ridge (도메인 FE 342개)")

    lgbm_params = dict(
        n_estimators=300,
        learning_rate=0.08,
        num_leaves=63,
        n_jobs=-1,
        random_state=42,
        verbose=-1
    )

    lgbm_raw = MultiOutputRegressor(LGBMRegressor(**lgbm_params)).fit(R_raw[train_idx], Y_all[train_idx])
    evaluate_predictions(Y_all[valid_idx], lgbm_raw.predict(R_raw[valid_idx]), "2-A. LightGBM (원본 226개)")

    lgbm_fe = MultiOutputRegressor(LGBMRegressor(**lgbm_params)).fit(X_fe[train_idx], Y_all[train_idx])
    evaluate_predictions(Y_all[valid_idx], lgbm_fe.predict(X_fe[valid_idx]), "2-B. LightGBM (도메인 FE 342개)")

[1/4] train.csv 로딩 중...
[2/4] 건너뛰기 분할(Interleaved Split) 적용 중...
  - Train 샘플 수: 50,625개 (비율: 6.2%)
  - Valid 샘플 수: 65,665개 (비율: 8.1%)
  - Test  샘플 수: 590,991개 (비율: 73.0%)
  - 외삽(300nm) 수: 102,719개 (비율: 12.7%)
[3/4] 반도체 광학·주파수 도메인 피처 추출 중...
  - 피처 생성 완료: 원본 226개 -> 최종 349개 특성

[4/4] 건너뛰기 분할 Valid 셋 기준 피처 엔지니어링 효과 검증...
[0. 평균 예측(155nm)] 전체 MAE: 72.311 nm | L1(SiN): 72.41, L2(SiO2): 72.27, L3(SiN): 72.34, L4(SiO2): 72.22
[1-A. Ridge (원본 226개)] 전체 MAE: 72.373 nm | L1(SiN): 72.78, L2(SiO2): 72.25, L3(SiN): 72.39, L4(SiO2): 72.07
[1-B. Ridge (도메인 FE 342개)] 전체 MAE: 54.019 nm | L1(SiN): 49.85, L2(SiO2): 62.73, L3(SiN): 46.62, L4(SiO2): 56.88
[2-A. LightGBM (원본 226개)] 전체 MAE: 42.554 nm | L1(SiN): 36.67, L2(SiO2): 52.66, L3(SiN): 41.30, L4(SiO2): 39.58
[2-B. LightGBM (도메인 FE 342개)] 전체 MAE: 30.371 nm | L1(SiN): 24.44, L2(SiO2): 42.40, L3(SiN): 30.29, L4(SiO2): 24.35
